<a id="unit-u.308c8da0636b4b9682186e5bd622a815"></a>

<div>
  <div>4-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Iceberg кестесі мен оның файлдарын сұрау</div>
</div>

1-модульдің он тарихи тапсырысы шағын жергілікті деректер көлінің Iceberg кестесінде де бар: S3-үйлесімді объектілік сақтау Silo оның файлдарын, ал Iceberg REST Catalog қызметі метадеректерін сақтайды. Doris-ті оған Catalog арқылы қосып, кестені табасыз, snapshot-тары мен деректер файлдарын оқисыз, сұрайсыз, Doris тұтынушы кестесімен біріктіресіз және ішкі кестеге жүктейсіз: бір рет Catalog арқылы, бір рет S3 кестелік функциясымен деректер файлынан. Әр тәсіл он тапсырыс және 12220.60 беруі керек. Шамамен 30 минут жоспарлаңыз.

**4-модуль: сыртқы деректерді сұрау және деректер көлі кестелерін біріктіру** · [Сабақ](course.md) · [Тест](quiz04_internal_files_and_lake_tables.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.f0b0b5f948404b1594fe9fc82f281023"></a>

<a id="prepare-the-lake"></a>

## Деректер көлін дайындау

Алдымен 1-зертханалық жұмысты аяқтаңыз. Келесі ұяшық курстың Doris жүйесіне қосылып, екі курс контейнерін — Silo және Iceberg REST Catalog — жергілікті 51900 және 51818 порттарында (`DW_LAKE_S3_PORT` және `DW_LAKE_REST_PORT` айнымалылары) іске қосады; алдымен екі портты тексеріп, біреуі бос болмаса, тоқтайды, портты атайды және оны босату не басқасын таңдау тәсілін көрсетеді. Алғаш іске қосқанда екі бейне жүктеледі; кейінгі орындаулар томдардағы деректерді қайта пайдаланады. Курстың README файлындағы [Тазалау](../../README.md#clean-up) бөлімі оларды тоқтату мен жоюды түсіндіреді.

Дайындық Doris ішінде зертханалық дерекқорыңыздың атына `_lake` қосып аталған Catalog, ал оның ішінде `orders` Iceberg кестесін жасайды; алғаш орындағанда он тапсырысты енгізеді. **Бұл деректер көлі кестесіне Doris жазады**: курста басқа қозғалтқыш жоқ. Нақты деректер көлінде мұндай кестелерді әдетте Spark, Flink не басқа жүйе жазады, ал Doris оларды сондай Catalog арқылы оқиды. Аяқтағанда ұяшық орындаған SQL-ді көрсетеді. Жұмыс тек `customers_sample`, `orders_from_lake` және `orders_from_file` ішкі кестелерін қайта жасайды.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.lakehouse import prepare_lakehouse
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows, sample
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()
source = prepare_lakehouse(lab, start=True)

<a id="unit-u.6f20dac31a5a47a68aad564e2b870373"></a>

**Күтілетін нәтиже**

Алты жасыл қадамы және көлдегі он тапсырысты тексеруі бар панель, содан кейін дайындық орындаған SQL: REST қызметінің `http://course-lake-rest:8181` мекенжайы, `s3://course-warehouse/` қоймасы және объектілік сақтаудың `http://course-lake-silo:9000` мекенжайы мен кілттері бар `CREATE CATALOG IF NOT EXISTS dw_course_l1_demo_lake`; сол Catalog ішінде `CREATE DATABASE` және `CREATE TABLE`; алғаш орындағанда он тапсырыстың `INSERT` операторы немесе кейінгі орындауда кестеде олар бар екені туралы түсініктеме. Бұл мекенжайлар — Doris жұмыс істейтін курс Docker желісіндегі атаулар: ноутбуктен сол қызметтер 127.0.0.1:51900 және 127.0.0.1:51818 арқылы жауап береді.

Ұяшық кестенің толық `dw_course_l1_demo_lake.dw_course_l1_demo.orders` атауын `source` ішінде сақтайды (`DW_DATABASE` орнатсаңыз, өз дерекқор атыңызбен).


<a id="unit-u.977e628388c64d8c8d8c9d7ea453c7fb"></a>

<a id="1-find-the-table-through-the-catalog"></a>

## 1. Кестені Catalog арқылы табу

**Catalog** — Doris-тің басқа жүйенің метадеректеріне кіретін нүктесі: Doris-тің өз кестелері `internal` Catalog-ында, ал деректер көлі кестесі дайындық жасаған Catalog-та. `internal` сыртындағы кесте `catalog.database.table` деп аталады. `SHOW CATALOGS`, `SHOW DATABASES FROM` және `SHOW TABLES FROM` осы жолмен төмен өтеді, ал `SHOW CREATE TABLE` кестенің түрін және файлдарының орнын көрсетеді.


In [ ]:
catalog, database, _ = source.split(".")
lab.sql("SHOW CATALOGS", title="The Catalogs of this Doris", columns=["CatalogName", "Type", "IsCurrent", "Comment"])
lab.sql(f"SHOW DATABASES FROM {catalog}", title="The databases in the lake Catalog")
lab.sql(f"SHOW TABLES FROM {catalog}.{database}", title="The tables in the lake database")
definition = lab.sql(f"SHOW CREATE TABLE {source}", title="What kind of table it is")
expect("ENGINE=ICEBERG_EXTERNAL_TABLE" in definition.iloc[0, 1], True, title="The table is an Iceberg table in the lake")

<a id="unit-u.96ae413c53aa4d52bb95343ef471781b"></a>

**Күтілетін нәтиже**

`SHOW CATALOGS` ағымдағы, түрі `internal` болатын `internal` және түрі `iceberg` болатын `dw_course_l1_demo_lake` Catalog-тарын тізеді; басқа зертханалық дерекқорлар үшін жасаған Catalog-тар да көрінеді. Деректер көлі Catalog-ында `dw_course_l1_demo`, `information_schema` және `mysql` дерекқорлары, ал дерекқорда бір `orders` кестесі бар. Оның анықтамасында `ENGINE=ICEBERG_EXTERNAL_TABLE` және `LOCATION 's3://course-warehouse/dw_course_l1_demo/orders'`, сондай-ақ `"write.parquet.compression-codec" = "zstd"` сияқты Iceberg қасиеттері көрсетілген, тексеру жасыл. Бағандар курстың ішкі кестелеріндегі `NOT NULL` бағандарынан өзгеше, әдеттегі Iceberg бағандары сияқты NULL қабылдайды.


<a id="unit-u.23e9792a9ea445469abf60205d05c0ab"></a>

<a id="2-read-the-tables-snapshots-and-files"></a>

## 2. Кестенің snapshot-тары мен файлдарын оқу

Iceberg кестесі — деректер файлдарының үстіндегі метадеректер. Әр бекіту, мысалы `INSERT`, жаңа файлдар және жаңа **snapshot**, яғни сол сәттегі кестені құрайтын файлдар тізімін жазады. Сұрау ағымдағы snapshot файлдарын оқиды. Doris Iceberg метадеректерін атауы кесте атауына жұрнақ қосу арқылы жасалған кестелер ретінде көрсетеді: `$snapshots` snapshot-тарды, ал `$files` ағымдағы snapshot деректер файлдарын тізеді.


In [ ]:
lab.sql(f"SELECT committed_at, snapshot_id, parent_id, operation FROM {source}$snapshots",
        title="The snapshots of the lake table")
files = lab.sql(f"SELECT file_path, file_format, record_count, file_size_in_bytes FROM {source}$files",
                title="The data files of the current snapshot")
data_file = files["file_path"][0]

<a id="unit-u.fb7ca79ea9724bb78e44fa640f0d4ea3"></a>

**Күтілетін нәтиже**

`operation` мәні `append`, ата-анасы жоқ бір snapshot: дайындықтың алғашқы орындалуындағы `INSERT`. Бір деректер файлы — `s3://course-warehouse/dw_course_l1_demo/orders/data/<random name>.zstd.parquet`, пішімі `PARQUET`, 10 жазба, шамамен 4.3 KB. Ұяшық 5-қадам үшін оның жолын `data_file` ішінде сақтайды. Басқа `INSERT` екінші snapshot пен екінші файл қосар еді; содан кейін Doris екеуін де оқыр еді.


<a id="unit-u.2129d32cb8da41a4aa880ce8a004f84e"></a>

<a id="3-query-the-lake-table-directly"></a>

## 3. Деректер көлі кестесін тікелей сұрау

Catalog арқылы Doris кесте метадеректерін REST қызметінен оқиды, қай файлдарды оқитынын жоспарлайды, оларды Silo-дан оқып, өз BE-сінде есептейді. Нәтиже тек Doris жадында болады: ішкі кестеде ештеңе сақталмайды. Он тапсырыс және 12220.60 күтіледі.


In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10, "12220.60")],
       title="The lake table holds ten orders, 12220.60")
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1", title="The plan of a lake table query");
lab.sql(f"SELECT order_id, customer_id, order_date, order_amount FROM {source} ORDER BY order_id",
        title="Ten orders in the lake");

<a id="unit-u.8a08ae71c792427d8fa4f1d5d5f8e949"></a>

**Күтілетін нәтиже**

Жасыл белгі, содан кейін жоспар: оның сканерлеуі — кестенің толық атауы, `order_id = 1` предикаты, `inputSplitNum=1, totalFileSize=4349` және сол шартты қамтитын `icebergPredicatePushdown` бар `VICEBERG_SCAN_NODE`: Doris сүзгіні Iceberg-ке береді, ол баған статистикасы сәйкес келмейтінін көрсететін файлдарды өткізіп жібере алады. Содан кейін 1–5 және 80–84 он тапсырысы көрсетіледі.


<a id="unit-u.d55b77be567d4aa98bd55a2d56bf18f8"></a>

<a id="4-join-an-internal-table-and-load-the-lake-table"></a>

## 4. Ішкі кестемен біріктіру және деректер көлі кестесін жүктеу

Doris ішінде әр тұтынушыға бір жолы бар `customers_sample` Unique Key кестесін жасап, көлдегі тапсырыстарды оған `customer_id` арқылы біріктіріңіз: әр тапсырыс дәл бір тұтынушыға сәйкес келеді, сондықтан біріктіру он жол мен 12220.60 сомасын сақтайды. Содан кейін тапсырыстың алты бағанын `INSERT INTO ... SELECT` арқылы `orders_from_lake` ішкі кестесіне жүктеп, деректер көлі кестесімен әр жол бойынша салыстырыңыз. Ішкі кесте — бүгінгі көл деректерінің көшірмесі: ол көлдегі кейінгі өзгерістерге ілеспейді.


In [ ]:
lab.execute("DROP TABLE IF EXISTS customers_sample")
lab.execute('CREATE TABLE customers_sample (customer_id BIGINT, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("customers_sample", ["customer_id", "customer_name"],
           [(r["customer_id"], r["customer_name"]) for r in sample()["customers"]])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN customers_sample c ON o.customer_id=c.customer_id"),
       [(10, "12220.60")], title="The join keeps ten orders, 12220.60")
lab.execute("DROP TABLE IF EXISTS orders_from_lake")
ddl = order_ddl("orders_from_lake")
show_sql("CREATE TABLE SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_from_lake ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_lake ORDER BY order_id"),
       history_rows(), title="orders_from_lake matches the lake table row by row")
lab.sql(f"""SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, ENGINE FROM {catalog}.information_schema.tables WHERE TABLE_NAME = 'orders'
UNION ALL
SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, ENGINE FROM internal.information_schema.tables
WHERE TABLE_SCHEMA = '{lab.database}' AND TABLE_NAME = 'orders_from_lake'""", title="The lake table and its internal copy");

<a id="unit-u.8e44bd2c119b4e7c99ac6bfea0105d66"></a>

**Күтілетін нәтиже**

`orders_from_lake` кесте анықтамасы, екі жасыл белгі және екі жол: көл Catalog-ындағы `ENGINE` мәні `iceberg` болатын `orders` көл кестесі және `internal` ішіндегі `ENGINE` мәні `Doris` болатын `orders_from_lake`. Деректер бірдей; оларды кім сақтайтыны өзгеше.


<a id="unit-u.9db3db3129744e2a8d700c0053616525"></a>

<a id="5-read-the-data-file-without-its-table-the-s3-table-function"></a>

## 5. Деректер файлын кестесінсіз оқу: S3 кестелік функциясы

**Кестелік функция** кесте сияқты жолдар қайтарады. `S3(...)` S3-үйлесімді объектілік сақтаудағы файлдарды оқиды: сіз бір файлдың не `*` арқылы бірнеше файлдың жолын, пішімін, объектілік сақтау мекенжайы мен кілттерін бересіз, ол дәл сол файлдарды оқиды. Оны 2-қадамдағы деректер көлі кестесінің деректер файлына бағыттаңыз. `DESC FUNCTION` Doris Parquet файлының өзінен оқитын бағандарды көрсетеді. Содан кейін файлды `INSERT INTO ... SELECT` арқылы `orders_from_file` ішіне жүктеп, салыстырыңыз.

Функция Iceberg туралы ештеңе білмейді: snapshot-тарды оқымайды. Деректер көлі кестесі кейін өзгерсе, мысалы жаңа snapshot-та жолдар жойылса, бұл ескі файлда олар әлі қалар еді; Catalog ағымдағы snapshot файлдарын оқиды.


In [ ]:
from dw_course.lakehouse import ACCESS_KEY, SECRET_KEY
file_function = f'''S3(
    "uri" = "{data_file}",
    "format" = "parquet",
    "s3.endpoint" = "http://course-lake-silo:9000", "s3.region" = "us-east-1",
    "s3.access_key" = "{ACCESS_KEY}", "s3.secret_key" = "{SECRET_KEY}",
    "use_path_style" = "true"
)'''
lab.sql(f"DESC FUNCTION {file_function}", title="The columns of the Parquet file", columns=["Field", "Type", "Null"])
lab.sql(f"SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM {file_function}",
        title="The file read directly")
lab.execute("DROP TABLE IF EXISTS orders_from_file")
lab.execute(order_ddl("orders_from_file"))
load_file = f"INSERT INTO orders_from_file ({','.join(ORDER_COLUMNS)})\nSELECT {','.join(ORDER_COLUMNS)}\nFROM {file_function}"
show_sql("Load the file into an internal table", load_file)
lab.execute(load_file)
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_file ORDER BY order_id"),
       history_rows(), title="orders_from_file matches the lake table row by row")

<a id="unit-u.7e69826317ee418d98f3fcfbf3f504b8"></a>

**Күтілетін нәтиже**

Файлдан алты баған, Doris Parquet-тен оқыған түрлер: `bigint`, `bigint`, `date`, `decimal(18,2)`, `int` және `text`, бәрі NULL қабылдайды. Файлды тікелей оқу 10 тапсырыс және 12220.60 береді, `INSERT INTO ... SELECT` көрсетіледі, тексеру жасыл: файл — кестенің деректері. `use_path_style` Silo күткендей бакет атауын URL жолына қояды.


<a id="unit-u.88d290c85e5946e6bf2009cd57c13337"></a>

<a id="6-practice-the-second-day-with-missing-customers"></a>

## 6. Жаттығу: екінші күн және жетіспейтін тұтынушылар

Деректер көліндегі 2013-01-02 тапсырыстары үшін `LEFT JOIN` қолданып, тапсырыстарды санаңыз, сомаларын қосыңыз және тұтынушысы `customers_sample` кестесінде жоқ тапсырыстарды санаңыз. 5 тапсырыс, 8276.40 және 0 жетіспейтін тұтынушы күтіледі. Содан кейін деректер көлі кестесі мен `orders_from_lake` кестесіндегі бірдей күн сүзгісінің жоспарларын салыстырыңыз: әрқайсысын қай сканерлеу түйіні оқиды және әрқайсысы не оқитынын айтады?


In [ ]:
# Your code here

<a id="unit-u.af2759d44919476e8754291c7ab1c27d"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
query = f"""SELECT COUNT(*) AS orders, SUM(o.order_amount) AS amount,
SUM(CASE WHEN c.customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customers
FROM {source} o LEFT JOIN customers_sample c ON o.customer_id=c.customer_id
WHERE o.order_date='2013-01-02'"""
lab.sql(query, title="Customer matches for the second day's lake orders");
expect(lab.query(query), [(5,"8276.40",0)], title="Five orders, 8276.40, no customer missing")
lab.sql("EXPLAIN SELECT * FROM orders_from_lake WHERE order_date='2013-01-02'", title="Internal table scan");
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_date='2013-01-02'", title="Lake table scan");

<a id="unit-u.d13fa46ea78945c492a0af72314b5d1d"></a>

**Күтілетін нәтиже**

Бір жол: 5 тапсырыс, 8276.40, 0 жетіспейтін тұтынушы; `LEFT JOIN` тұтынушысы жоқ тапсырысты сақтап, тұтынушы бағандарын `NULL` қалдырады, сондықтан бұл есеп оны табар еді. Ішкі кесте жоспары `VOlapScanNode`, `partitions=1/1` және `tablets=1/1` арқылы Doris сақтайтын tablet-ті сканерлейді. Деректер көлі кестесінің жоспары `VICEBERG_SCAN_NODE`, `inputSplitNum=1` және `totalFileSize=4349` арқылы кесте метадеректерімен табылған объектілік сақтаудағы файлды сканерлейді, күн сүзгісі Iceberg-ке беріледі.


<a id="unit-u.543f164c93ca4d28a59203f437174a61"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- деректер көлі кестесінің Catalog-ын, дерекқорын және кестесін атай аласыз, `ENGINE=ICEBERG_EXTERNAL_TABLE` мен оның `LOCATION` мәні нені білдіретінін айтасыз;
- snapshot-ын және деректер файлын оқып, кесте сұрауы неге тек ағымдағы snapshot файлдарын оқитынын түсіндіре аласыз;
- деректер көлі кестесі, біріктіру, `orders_from_lake` және `orders_from_file` барлығы он тапсырыс және 12220.60 береді, әрі олардың қайсысын Doris сақтайтынын айта аласыз.


<a id="unit-u.3692068897084aaea9c6b1cff003b4eb"></a>

<a id="next"></a>

## Келесі қадам

[4-тест: сыртқы деректерді сұрау және деректер көлі кестелерін біріктіру](quiz04_internal_files_and_lake_tables.ipynb) файлын ашыңыз, содан кейін [5-модуль: деректерді пакеттік және үздіксіз жүктеу](../module05-ingestion/course.md) бөліміне өтіңіз.

Деректер көлін тоқтатып, деректерін сақтау үшін курс каталогынан мынаны орындаңыз:

```bash
docker compose --project-name doris-warehousing-lake --file environments/lakehouse/compose.yml stop
```

Жалғастыру үшін осы жұмысты баптауынан бастап **Деректер көлін дайындау** бөліміне дейін қайта орындаңыз. 4–5-модульдердің кеңейтілген жұмысы сол көл деректерін қолданады; оны аяқтағанша томдарды сақтаңыз.
